In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Classic Bengio style MLP

In [10]:
class MLP(nn.Module):
    def __init__(self, vocab_size,context_length,embedding_dim,hidden_dim):
        super().__init__()
        self.h_in = embedding_dim*context_length
        self.emb_layer = nn.Embedding(vocab_size,embedding_dim)
        self.h = nn.Linear(embedding_dim*context_length,hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(hidden_dim,vocab_size)

        

    def forward(self,x):
        x = self.emb_layer(x).reshape(-1,self.h_in)
        x = self.h(x)
        x = self.bn(x)
        x = self.tanh(x)
        o = self.out(x)
        return o


My variant with position embedding

In [15]:
class MLP_PE(nn.Module):
    def __init__(self,vocab_size,context_length,word_embedding_dim,hidden_dim,max_pos,pos_embedding_dim):
        """
        (B,C) → (B, C*E + P) → (B, H) → (B, V)
        """
        super().__init__()
        self.max_pos = max_pos
        self.w_in =  word_embedding_dim*context_length
        self.p_in = pos_embedding_dim
        self.h_in =self.w_in + self.p_in
        self.word_emb_layer = nn.Embedding(vocab_size,word_embedding_dim)
        self.pos_emb_layer = nn.Embedding(max_pos,pos_embedding_dim)
        self.h = nn.Linear(self.h_in,hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(hidden_dim,vocab_size)

    def forward(self,x,pos):
        assert pos.shape[0] == x.shape[0]
        pos = torch.clamp(pos,max=self.max_pos -1)
        x_w = self.word_emb_layer(x).reshape(-1,self.w_in) # -> (B,C*E)
        x_p = self.pos_emb_layer(pos) # -> (B,P)
        x = torch.cat((x_w,x_p),dim=1) # -> (B, C*E + P)
        x = self.h(x)
        x = self.bn(x)
        x = self.tanh(x)
        return self.out(x)


In [ ]:
MLP_model = MLP(27,4,10,200)

In [28]:
PosMLP = MLP_PE(27,4,10,172,15,10)

In [13]:
sum(p.numel() for p in MLP_model.parameters())

14297

In [29]:
sum(p.numel() for p in PosMLP.parameters())

14207